# Notebook 05 (Oscar): MLP Training -- Exp 3 and Exp 4

**Exp 3:** Delta Residue + Full Wildtype -- `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Exp 4:** Delta Sequence -- `mean_pool(mutant_seq) - mean_pool(wt_seq)` (cached)

Both experiments run for both ESM-2 and AbLang2. Training uses MSE loss.
Evaluation metric: Spearman correlation per dataset and aggregate (excluding HER2 separately).
All runs logged to W&B.

## Setup

In [11]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Environment: local
Repo: /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project


Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [12]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

Drive root:      /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project
Embedding dir:   /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/embeddings
Checkpoint dir:  /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/checkpoints
Paths set.


`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [13]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

Local run -- installation skipped.


In [14]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Autoreload enabled.


In [15]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device: mps
Apple MPS -- Apple Silicon unified memory


Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [16]:
import numpy as np
import pandas as pd
import wandb
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

Imports OK.


## Data Loading and Splits

In [17]:
df = load_abagym_antibody(DATA_DIR)

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")
print()

# Verify all datasets are represented in each split
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    counts = df.iloc[idx]['DMS_name'].value_counts().to_dict()
    print(f"{split_name}: {counts}")

Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}


Loads the AbAgym metadata CSV and creates the stratified 80/10/10 split.

The split is stratified within each antibody dataset separately, then pooled.
This ensures all 5 antibodies are represented in train, val, and test.
The `random_state=42` is fixed -- Lucas uses the same value in NB05_lucas.ipynb
to guarantee identical splits across both notebooks.

Confirmed output:

```
Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val:   {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test:  {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
```

All 5 antibodies present in every split. HER2 val/test N=18 -- Spearman on 18 samples
is noisy; report but note unreliability. Splits are identical to Lucas's notebook.

## Experiment 4: Delta Sequence

**Input:** `mean_pool(mutant_seq) - mean_pool(wt_seq)`  
**Dims:** ESM-2 = 2560, AbLang2 = 960  
**Owner:** Oscar

The simplest and most direct embedding strategy. The delta sequence vector captures
the antibody-wide shift in embedding space caused by the mutation. From NB04 EDA,
the L2 norm of this vector is already weakly predictive of mutation effect
(Spearman r=0.083 ESM-2, r=0.216 AbLang2) without any supervised training.
A trained MLP operating on the full 2560/960-dim vector has access to directional
information that the norm discards, so performance should improve substantially.

In [20]:
# Build datasets for both models -- Exp 4 (DELTA_SEQUENCE)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_SEQUENCE,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_SEQUENCE: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

esm2 DELTA_SEQUENCE: input_dim=2560, label=0.7380, region=CDR_H3
ablang2 DELTA_SEQUENCE: input_dim=960, label=0.7380, region=CDR_H3


Sanity check: verifies that the dataset loads correctly and the input dimension
matches expectations (ESM-2=2560, AbLang2=960).

Confirmed output:

```
esm2 DELTA_SEQUENCE: input_dim=2560, label=0.7380, region=CDR_H3
ablang2 DELTA_SEQUENCE: input_dim=960, label=0.7380, region=CDR_H3
```

Input dims correct. Both models return the same label and region for row 0
(Ang2_2017_G6 H:P100A, CDR_H3, MinMax score=0.7380).

## Experiment 3: Delta Residue + Full Wildtype

**Input:** `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Dims:** ESM-2 = 1280 + 2560 = 3840, AbLang2 = 480 + 960 = 1440  
**Owner:** Oscar

Combines the local mutation signal (per-token delta at the mutation site) with
global antibody context (wildtype sequence embedding). The rationale: the per-token
delta alone tells you how much the mutation changed that position, but the wildtype
context tells the MLP what kind of antibody this is (CDR vs FR context, scaffold
type, chain identity). Together they provide both local and global information.

The wildtype embedding is shared across all mutations of the same antibody --
the dataset class handles the index lookup internally.

In [22]:
# Build datasets for both models -- Exp 3 (DELTA_RESIDUE_PLUS_WILD)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_RESIDUE_PLUS_WILD: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

esm2 DELTA_RESIDUE_PLUS_WILD: input_dim=3840, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE_PLUS_WILD: input_dim=1440, label=0.7380, region=CDR_H3


Sanity check: verifies input dimensions for Exp 3
(ESM-2=3840, AbLang2=1440) and that the wildtype index lookup works.

Confirmed output:

```
esm2 DELTA_RESIDUE_PLUS_WILD: input_dim=3840, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE_PLUS_WILD: input_dim=1440, label=0.7380, region=CDR_H3
```

Input dims correct (1280 delta_residue + 2560 wt_sequence for ESM-2;
480 + 960 for AbLang2). Wildtype index inversion verified -- DMS_name lookup
returns the correct wt_sequence row.

## Experiment 4: Training

Run for both ESM-2 and AbLang2. Each call to `train_abagym` logs a separate
W&B run. Results are stored in `exp4_results` for downstream test evaluation.

**Expected input dims:** ESM-2 = 2560, AbLang2 = 960  
**Architecture:** [256, 128] hidden layers, ReLU + Dropout(0.1)  
**Early stopping:** patience = 10 epochs on aggregate val Spearman

In [27]:
exp4_results = {}

for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_SEQUENCE,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    # Infer input_dim from one sample
    x0, _, _ = ds_full[0]
    input_dim = x0.shape[0]
    print(f"{model_name} DELTA_SEQUENCE -- input_dim={input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy='delta_sequence',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    exp4_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print(f"  Per-dataset: {result['per_dataset_spearman']}")
    print()

esm2 DELTA_SEQUENCE -- input_dim=2560


esm2_delta_sequence_lambda0.0:  33%|███▎      | 33/100 [00:10<00:21,  3.06epoch/s, best=0.6381, patience=9/10, train_mse=0.0175, val_rho=0.6225]

Early stopping at epoch 34. Best epoch: 24 (val ρ=0.6381)


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇███
train_mse,█▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_spearman,▁▃▅▅▆▆▇▇▇▇▇▇▇█▇▇▇█████████▇█▇█▇███
val_spearman/Ang2_2017_G6,▁▅▆▇▇▇█▆▇█▇▇▇█▆▇▇▆▇█▆▇▇▇▇▇▆▇▆▇▇▇▇▆
val_spearman/EGFR_2013_Cetuximab,▁▂▄▄▅▅▆▆▇▇▇▇▇▇▇█▇█████████████████
val_spearman/HER2_2021_trastuzumab,▁▆█▇▇▆▅▅▅▅▅▄▅▅▆▆▅▆▇▇▆▇▆▆▇▇▇▇█▇▇██▇
val_spearman/VEGF_2017b_G6,▁▂▄▄▅▅▆▆▇▆▆▆▆▇▆▆▆▇▇▆▇▇▆█▇▇▇▇▇█▆▇▇▇
val_spearman/lysozyme_2019_D441,▁▂▃▃▅▅▆▇▇███▇█▇▇██▇███▇██▇▇▇▇▇▇▇▇▇
val_spearman_HER2,▁▆█▇▇▆▅▅▅▅▅▄▅▅▆▆▅▆▇▇▆▇▆▆▇▇▇▇█▇▇██▇
val_spearman_excl_her2,▁▃▄▅▆▆▇▇▇▇▇▇▇█▇█▇█████████▇█▇█▇███
best_epoch,24


  Best epoch: 24
  Best val Spearman (all): 0.6381
  Per-dataset: {'Ang2_2017_G6': 0.7034044637665814, 'EGFR_2013_Cetuximab': 0.7258646713232709, 'HER2_2021_trastuzumab': 0.43704725422550716, 'VEGF_2017b_G6': 0.4920710149093832, 'lysozyme_2019_D441': 0.6049376143939758}

ablang2 DELTA_SEQUENCE -- input_dim=960


ablang2_delta_sequence_lambda0.0:  44%|████▍     | 44/100 [00:13<00:17,  3.24epoch/s, best=0.6552, patience=9/10, train_mse=0.0101, val_rho=0.6322]

Early stopping at epoch 45. Best epoch: 35 (val ρ=0.6552)


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train_mse,█▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_spearman,▁▃▄▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇███▇█▇████████████████
val_spearman/Ang2_2017_G6,▁▃▄▅▆▇▆▇▇▇▇▇▇▇█▇▇▇▇████▇████████████████
val_spearman/EGFR_2013_Cetuximab,▁▃▅▅▅▆▅▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇█▇▇▇███▇▇█▇██▇███▇
val_spearman/HER2_2021_trastuzumab,▆██▇▇▆▆▅▄▄▅▅▄▃▅▄▄▃▃▃▁▂▃▂▃▃▂▃▃▂▄▃▄▃▄▃▄▄▃▂
val_spearman/VEGF_2017b_G6,▁▂▄▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇█▇▇█▇▇▇▇▇█▇▇▇██▇█████▇
val_spearman/lysozyme_2019_D441,▁▃▂▃▄▅▅▅▆▆▆▇▆▆▇▆▇▇▇▇▇▇█▇███▇█████████▇█▇
val_spearman_HER2,▆██▇▇▆▆▅▄▄▅▅▄▃▅▄▄▃▃▃▁▂▃▂▃▃▂▃▃▂▄▃▄▃▄▃▄▄▃▂
val_spearman_excl_her2,▁▃▄▄▅▆▆▆▇▇▇▇▇▇▇▇▇██████▇████████████████
best_epoch,35


  Best epoch: 35
  Best val Spearman (all): 0.6552
  Per-dataset: {'Ang2_2017_G6': 0.7980242298802351, 'EGFR_2013_Cetuximab': 0.6540204231721839, 'HER2_2021_trastuzumab': 0.39627011806571105, 'VEGF_2017b_G6': 0.6651364169392133, 'lysozyme_2019_D441': 0.5412742917833512}



Confirmed val results:

```
esm2 DELTA_SEQUENCE -- input_dim=2560
  Best epoch: 24
  Best val Spearman (all): 0.6381
  Per-dataset: Ang2=0.7034, EGFR=0.7259, HER2=0.4370, VEGF=0.4921, lysozyme=0.6049

ablang2 DELTA_SEQUENCE -- input_dim=960
  Best epoch: 35
  Best val Spearman (all): 0.6552
  Per-dataset: Ang2=0.7980, EGFR=0.6540, HER2=0.3963, VEGF=0.6651, lysozyme=0.5413
```

Both models far exceed the EDA norm baseline (ESM-2: 0.083→0.638, AbLang2: 0.216→0.655).
The full directional delta vector carries substantially more signal than the norm alone.

AbLang2 leads overall (0.655 vs 0.638) but the gap is smaller than EDA suggested -- the
MLP recovers signal from ESM-2's delta space that the scalar norm misses. Dataset split:
ESM-2 leads on EGFR (0.726 vs 0.654) and lysozyme (0.605 vs 0.541), the FR-heavy datasets.
AbLang2 leads on Ang2 (0.798 vs 0.703) and VEGF (0.665 vs 0.492), the CDR-heavy G6 datasets.

## Experiment 4: Test Evaluation

Evaluate the best-epoch model (restored by `train_abagym`) on the held-out
test split. Report per-dataset Spearman, aggregate (all 5), and aggregate
excluding HER2.

In [28]:
print("=== Experiment 4: DELTA_SEQUENCE -- Test Results ===")
for model_name, result in exp4_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()}")
    print(f"  Aggregate Spearman (all 5):   {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

=== Experiment 4: DELTA_SEQUENCE -- Test Results ===

ESM2
  Aggregate Spearman (all 5):   0.6122
  Aggregate Spearman (excl HER2): 0.6026
  HER2 Spearman:                0.8262
  Per-dataset:
    Ang2_2017_G6                        0.6470
    EGFR_2013_Cetuximab                 0.6604
    HER2_2021_trastuzumab               0.8262
    VEGF_2017b_G6                       0.5416
    lysozyme_2019_D441                  0.5632

ABLANG2
  Aggregate Spearman (all 5):   0.6024
  Aggregate Spearman (excl HER2): 0.6034
  HER2 Spearman:                0.5418
  Per-dataset:
    Ang2_2017_G6                        0.7303
    EGFR_2013_Cetuximab                 0.5265
    HER2_2021_trastuzumab               0.5418
    VEGF_2017b_G6                       0.7451
    lysozyme_2019_D441                  0.5258


Confirmed test results:

```
ESM-2
  Aggregate Spearman (all 5):     0.6122
  Aggregate Spearman (excl HER2): 0.6026
  HER2 Spearman:                  0.8262
  Ang2_2017_G6:                   0.6470
  EGFR_2013_Cetuximab:            0.6604
  VEGF_2017b_G6:                  0.5416
  lysozyme_2019_D441:             0.5632

AbLang2
  Aggregate Spearman (all 5):     0.6024
  Aggregate Spearman (excl HER2): 0.6034
  HER2 Spearman:                  0.5418
  Ang2_2017_G6:                   0.7303
  EGFR_2013_Cetuximab:            0.5265
  VEGF_2017b_G6:                  0.7451
  lysozyme_2019_D441:             0.5258
```

Key findings:

Excluding HER2, the two models are essentially tied (ESM-2 0.603, AbLang2 0.603).
The dataset-level split is sharp: ESM-2 leads on EGFR (+0.134) and lysozyme (+0.037);
AbLang2 leads on Ang2 (+0.083) and VEGF (+0.204). This tracks the CDR/FR composition
of each dataset and the EDA finding that AbLang2 is more sensitive to CDR-heavy mutations
while ESM-2 is more sensitive to FR-heavy mutations.

ESM-2 HER2 test Spearman = 0.826 vs val = 0.437 -- a large jump. With N=18 test
samples, this is high-variance and should not be over-interpreted. The val number
(0.437) is the more reliable estimate. Report both and note the instability.

## Experiment 3: Training

Run for both ESM-2 and AbLang2. Same architecture and hyperparameters as Exp 4.

**Expected input dims:** ESM-2 = 3840, AbLang2 = 1440  
**Architecture:** [256, 128] hidden layers, ReLU + Dropout(0.1)

The larger input from Exp 3 means the first projection layer (3840→256 or
1440→256) compresses more aggressively. If Exp 3 doesn't clearly improve over
Exp 4, the wildtype context is not adding useful signal beyond what the delta
already encodes.

In [29]:
exp3_results = {}

for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    x0, _, _ = ds_full[0]
    input_dim = x0.shape[0]
    print(f"{model_name} DELTA_RESIDUE_PLUS_WILD -- input_dim={input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy='delta_residue_plus_wild',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    exp3_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print(f"  Per-dataset: {result['per_dataset_spearman']}")
    print()

esm2 DELTA_RESIDUE_PLUS_WILD -- input_dim=3840


esm2_delta_residue_plus_wild_lambda0.0:  87%|████████▋ | 87/100 [00:30<00:04,  2.83epoch/s, best=0.7006, patience=9/10, train_mse=0.0065, val_rho=0.6729]

Early stopping at epoch 88. Best epoch: 78 (val ρ=0.7006)


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇████
train_mse,█▇▇▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁
val_spearman,▁▃▄▄▅▅▅▆▆▆▆▆▆▆▇▆▇▇▇▇▇█▇▇▇█▇▇██▇█▇▇███▇█▇
val_spearman/Ang2_2017_G6,▁▅▅▄▅▅▅▃▆▅▅▄▆▅▅▅▆▆▇▇▇▇▇▇▆▇▇▇▇▇▇▇▇▆▇▇▇███
val_spearman/EGFR_2013_Cetuximab,▁█▇▇▇▅▆▆▆▅▅▇▆▅▃▅▆▆▅▅▅▅▃▆▅▇▆▇▆▅▅▇▅▇▄▅▄▅▅▄
val_spearman/HER2_2021_trastuzumab,▃▂▁▄▄▆▄▄▄▅▇▅▄▄▃▆▁▇▆▄▅▇▆▇▆██▇█▇▇▇▆█▇▇▇▇▇▆
val_spearman/VEGF_2017b_G6,▁▁▁▂▃▂▃▃▂▂▂▃▄▃▄▅▅▆▆▇▇▇▇▇▇▇█▇▇█▇▇▇▇▇▇▇█▇█
val_spearman/lysozyme_2019_D441,▁▂▄▅▅▅▆▆▆▆▇▆▇▇▇▇███▇▇▇▇███▇█▇█▇▇█▇█▇█▇▇█
val_spearman_HER2,▃▂▃▁▅▄▃▄▅▄▅▅▅▇▅▄▅▆▅▅▅▅▇▄▄▇▆▇▆▆███▅▆▆▆▆▆▆
val_spearman_excl_her2,▁▃▆▅▆▆▆▆▆▆▇▇▇▇▇▇▇▆▇▇▇▇▇█▇▇██████▇███████
best_epoch,78


  Best epoch: 78
  Best val Spearman (all): 0.7006
  Per-dataset: {'Ang2_2017_G6': 0.7899436661389305, 'EGFR_2013_Cetuximab': 0.611590401490324, 'HER2_2021_trastuzumab': 0.6388417998368059, 'VEGF_2017b_G6': 0.743690574608611, 'lysozyme_2019_D441': 0.63276337101528}

ablang2 DELTA_RESIDUE_PLUS_WILD -- input_dim=1440


ablang2_delta_residue_plus_wild_lambda0.0:  59%|█████▉    | 59/100 [00:19<00:13,  3.01epoch/s, best=0.6621, patience=9/10, train_mse=0.0068, val_rho=0.6513]

Early stopping at epoch 60. Best epoch: 50 (val ρ=0.6621)


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train_mse,█▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_spearman,▁▄▅▅▅▅▆▅▆▆▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇█▇▇█▇██▇▇██▇█▇
val_spearman/Ang2_2017_G6,▁▅▅▅▅▆▆▅▆▇▆▇▇▇▇█▆▇▇▆▇▇▇█▇██▇▇████▇▇█▇▇█▇
val_spearman/EGFR_2013_Cetuximab,▃▆▄█▅▆▇▃▃▄▂▁▄▁█▄▃▄▅▆▇▆▇▆▇▆█▆▅▅▅▆▅▅▇▅▄█▇▆
val_spearman/HER2_2021_trastuzumab,▂▁▂▅▄▅▇▅▅▄▅▅▅▄▅▅▅▅▇▇▇▇▅▆▅▄▇▇▆▇██▆▆▇▅▆▆█▆
val_spearman/VEGF_2017b_G6,▁▁▃▃▂▁▃▃▄▃▃▄▄▄▄▅▆▅▅▆▅▆▆▆▆▅▆▇▇▇▆▆▇▇▇▇█▇██
val_spearman/lysozyme_2019_D441,▁▂▃▄▅▆▇▆▆▇▆▆▇█▇▇▇▇▇▇▇█▇▆█▇▇███▇▇▇▇██▇█▇▇
val_spearman_HER2,▂▁▂▅▄▅▇▆▆▅▆▅▅▅▅▄▅█▅▅▇▄▇▇▅▇▄▇▆▇▇██▆▆▅▅▆▇█
val_spearman_excl_her2,▁▃▃▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇██▇▇▇▇▇█▇█▇███
best_epoch,50


  Best epoch: 50
  Best val Spearman (all): 0.6621
  Per-dataset: {'Ang2_2017_G6': 0.7807087361488677, 'EGFR_2013_Cetuximab': 0.6046927653832903, 'HER2_2021_trastuzumab': 0.3941789828780292, 'VEGF_2017b_G6': 0.6181685049300453, 'lysozyme_2019_D441': 0.6280469851231648}



Confirmed val results:

```
esm2 DELTA_RESIDUE_PLUS_WILD -- input_dim=3840
  Best epoch: 78
  Best val Spearman (all): 0.7006
  Per-dataset: Ang2=0.7899, EGFR=0.6116, HER2=0.6388, VEGF=0.7437, lysozyme=0.6328

ablang2 DELTA_RESIDUE_PLUS_WILD -- input_dim=1440
  Best epoch: 50
  Best val Spearman (all): 0.6621
  Per-dataset: Ang2=0.7807, EGFR=0.6047, HER2=0.3942, VEGF=0.6182, lysozyme=0.6280
```

Exp 3 improves over Exp 4 for both models (ESM-2: 0.638→0.701, AbLang2: 0.655→0.662).
ESM-2 now leads AbLang2 (0.701 vs 0.662). Adding wildtype context helps ESM-2 more,
likely because the global antibody embedding compensates for ESM-2's per-chain forward
pass -- AbLang2 already has cross-chain context from joint VH|VL processing.

ESM-2 best epoch = 78 vs 24 for Exp 4. The 3840-dim input takes longer to converge.
ESM-2 HER2 val = 0.639 (vs 0.437 in Exp 4) -- more consistent; wildtype context
stabilizes HER2 predictions.

## Experiment 3: Test Evaluation

In [30]:
print("=== Experiment 3: DELTA_RESIDUE_PLUS_WILD -- Test Results ===")
for model_name, result in exp3_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()}")
    print(f"  Aggregate Spearman (all 5):   {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

=== Experiment 3: DELTA_RESIDUE_PLUS_WILD -- Test Results ===

ESM2
  Aggregate Spearman (all 5):   0.6963
  Aggregate Spearman (excl HER2): 0.6946
  HER2 Spearman:                0.7068
  Per-dataset:
    Ang2_2017_G6                        0.8294
    EGFR_2013_Cetuximab                 0.6621
    HER2_2021_trastuzumab               0.7068
    VEGF_2017b_G6                       0.7815
    lysozyme_2019_D441                  0.5737

ABLANG2
  Aggregate Spearman (all 5):   0.6640
  Aggregate Spearman (excl HER2): 0.6693
  HER2 Spearman:                0.5189
  Per-dataset:
    Ang2_2017_G6                        0.7610
    EGFR_2013_Cetuximab                 0.5909
    HER2_2021_trastuzumab               0.5189
    VEGF_2017b_G6                       0.6802
    lysozyme_2019_D441                  0.6259


Confirmed test results:

```
ESM-2
  Aggregate Spearman (all 5):     0.6963
  Aggregate Spearman (excl HER2): 0.6946
  HER2 Spearman:                  0.7068
  Ang2_2017_G6:                   0.8294
  EGFR_2013_Cetuximab:            0.6621
  VEGF_2017b_G6:                  0.7815
  lysozyme_2019_D441:             0.5737

AbLang2
  Aggregate Spearman (all 5):     0.6640
  Aggregate Spearman (excl HER2): 0.6693
  HER2 Spearman:                  0.5189
  Ang2_2017_G6:                   0.7610
  EGFR_2013_Cetuximab:            0.5909
  VEGF_2017b_G6:                  0.6802
  lysozyme_2019_D441:             0.6259
```

Exp 3 vs Exp 4 test (excl HER2): ESM-2 0.695 vs 0.603 (+0.092), AbLang2 0.669 vs 0.603 (+0.066).
Wildtype context adds substantial value to both models. ESM-2 now clearly leads AbLang2
(0.695 vs 0.669 excl HER2).

ESM-2 leads on Ang2 (+0.068), EGFR (+0.071), VEGF (+0.101). AbLang2 leads only on
lysozyme (+0.052). The wildtype context appears to help ESM-2 more on CDR-heavy datasets
(Ang2, VEGF) -- previously AbLang2's domain -- suggesting the global wildtype embedding
gives ESM-2 the scaffold context it was missing from separate per-chain passes.

## Summary: Exp 3 vs Exp 4

In [31]:
rows = []
for exp_name, results_dict in [('Exp4_DeltaSeq', exp4_results), ('Exp3_DeltaRes+WT', exp3_results)]:
    for model_name, result in results_dict.items():
        metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
        row = {
            'Experiment': exp_name,
            'Model': model_name,
            'Spearman_all': round(metrics['aggregate'], 4),
            'Spearman_excl_HER2': round(metrics['exclude_her2'], 4),
            'HER2': round(metrics['HER2'], 4),
        }
        for ds, r in metrics['per_dataset'].items():
            row[ds] = round(r, 4)
        rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

      Experiment   Model  Spearman_all  Spearman_excl_HER2   HER2  Ang2_2017_G6  EGFR_2013_Cetuximab  HER2_2021_trastuzumab  VEGF_2017b_G6  lysozyme_2019_D441
   Exp4_DeltaSeq    esm2        0.6122              0.6026 0.8262        0.6470               0.6604                 0.8262         0.5416              0.5632
   Exp4_DeltaSeq ablang2        0.6024              0.6034 0.5418        0.7303               0.5265                 0.5418         0.7451              0.5258
Exp3_DeltaRes+WT    esm2        0.6963              0.6946 0.7068        0.8294               0.6621                 0.7068         0.7815              0.5737
Exp3_DeltaRes+WT ablang2        0.6640              0.6693 0.5189        0.7610               0.5909                 0.5189         0.6802              0.6259


Confirmed summary (Exp 4 vs Exp 3, test set):

| Experiment | Model | Spearman_all | Spearman_excl_HER2 | HER2 | Ang2 | EGFR | VEGF | lysozyme |
|---|---|---|---|---|---|---|---|---|
| Exp4_DeltaSeq | ESM-2 | 0.6122 | 0.6026 | 0.8262 | 0.6470 | 0.6604 | 0.5416 | 0.5632 |
| Exp4_DeltaSeq | AbLang2 | 0.6024 | 0.6034 | 0.5418 | 0.7303 | 0.5265 | 0.7451 | 0.5258 |
| Exp3_DeltaRes+WT | ESM-2 | 0.6963 | 0.6946 | 0.7068 | 0.8294 | 0.6621 | 0.7815 | 0.5737 |
| Exp3_DeltaRes+WT | AbLang2 | 0.6640 | 0.6693 | 0.5189 | 0.7610 | 0.5909 | 0.6802 | 0.6259 |

Exp 3 (delta_residue + wildtype) outperforms Exp 4 (delta_sequence) for both models.
Best result so far: ESM-2 Exp 3, test Spearman excl HER2 = 0.6946.

## Interpretation: Information Availability and Model Capacity

Important framing: the foundation models (ESM-2, AbLang2) are **frozen** throughout
all experiments. No fine-tuning occurs. The MLP regression head is trained from
scratch on top of pre-computed, fixed embeddings. When we say "ESM-2 performs better
in Exp 3", we mean the MLP trained on ESM-2's embeddings performs better -- the
foundation model itself does not change.

With that in mind, the two-experiment comparison reveals a pattern:
**the MLP head trained on ESM-2 embeddings scales more steeply with input information
than the MLP head trained on AbLang2 embeddings.**

| Strategy | ESM-2 MLP (excl HER2) | AbLang2 MLP (excl HER2) | ESM-2 lead |
|---|---|---|---|
| Exp 4: delta sequence only | 0.603 | 0.603 | 0.000 |
| Exp 3: delta residue + wildtype | 0.695 | 0.669 | +0.026 |

With minimal input (Exp 4), the two MLPs tie. With richer input (Exp 3), the ESM-2
MLP pulls ahead by 0.026.

**Why the ESM-2 MLP benefits more from the additional wildtype embedding:**

AbLang2 encodes global antibody context internally via joint VH|VL forward passes --
every token embedding is already contextualized by the full paired sequence. The
wildtype scaffold identity, chain relationships, and CDR/FR positional context are
all already present in the cached embedding before the MLP sees it. Providing the
explicit wildtype sequence embedding adds relatively less marginal information to a
head trained on AbLang2 features.

ESM-2 embeds H and L chains in separate forward passes with no cross-chain attention.
The delta residue embedding at the mutation site encodes local perturbation but lacks
global antibody identity in the input to the MLP. Adding the wildtype sequence
embedding gives the MLP the missing context: what kind of antibody this is and what
its full wildtype embedding looks like. The MLP can now condition its prediction on
both the local change and the global scaffold.

**Speculation:**

ESM-2 was pretrained on ~250M diverse protein sequences -- a far richer corpus than
AbLang2's OAS antibodies. This means ESM-2's embedding space encodes more varied
structural and functional protein features, but these are distributed across many
protein families. The MLP head must learn which dimensions are relevant to antibody
mutation effect.

When the input is minimal (delta sequence only), the antibody-specific signal in
ESM-2's embeddings is harder for the MLP to isolate. As the input becomes richer
(adding the wildtype embedding provides scaffold context), the MLP has more to work
with and can leverage ESM-2's broader representational capacity more effectively.

The domain-specific model (AbLang2) provides higher information density per embedding
dimension -- its features are already tuned to antibody-relevant variation. This
gives the AbLang2 MLP a floor advantage in low-information regimes. But ESM-2's
larger embedding space (2560 vs 960 dims) may contain more total signal once the
right input context is provided, giving it a higher ceiling.

Whether this pattern holds at the residue level (Exp 2, Lucas) is a direct test:
with only the single-token delta and no global context, we expect the AbLang2 MLP
to lead again -- consistent with the hypothesis that the ESM-2 MLP needs more
contextual input to close the gap.